# Tier extensions — open work after the 2026-06-01 Tier 1+2 expansion

Companion to [`model_overviews/crossmodal_pca_pls_closed_form_overview.ipynb`](model_overviews/crossmodal_pca_pls_closed_form_overview.ipynb).
That notebook closed Tier 1 (symmetric downstream + 10-seed cross-modal on every basis)
and Tier 2.2/2.3 (10-seed combined predictor + 10-seed identifiability). This notebook
implements the remaining open work:

| Tier | What | Status here |
|---|---|---|
| **2.1** | 10-seed BR robustness for `none`, `bv`, `demo` bases (PLS≈BR on `bv+demo` already verified single-seed) | ✅ runnable code below (~2h compute) |
| **3.4** | Cortical thickness as higher-rank anatomy basis — the most scientifically interesting open question | ✅ runnable code below (depends on data probe) |
| **3.1** | K_PCA dimension sensitivity sweep (K ∈ {128, 256, 512}) | ✅ runnable code below (~5 min) |
| **3.2** | K_PLS components sensitivity sweep (K ∈ {32, 64, 128}) | ✅ runnable code below (~5 min) |
| **3.3** | Demographics decomposition (age-only / sex-only / race_eth-only as separate residualization bases) | ✅ runnable code below (~15 min) |
| **4** | Generalization to other parcellations / cohorts | ❌ markdown stub only — needs data setup not in repo |
| **5** | Methodology refinements (cross-fit train predictions, hierarchical AUC CIs) | ❌ markdown stub only — not paper-blocking |

Each tier section is self-contained: cache-resumable loops where applicable, own CSV
outputs under `results/local_results/Tier*/`.


## How to use this notebook

**Two ways to bring this notebook online:**

### A. Warm kernel (recommended if you've already run the main notebook)
If you've just finished running `crossmodal_pca_pls_closed_form_overview.ipynb` in this
kernel (so `_base`, `_RESID`, `_BASES`, `_pca_pls_predict`, `_combined_predict`,
`_fit_basis_ols`, `_full_panel_eval` are all in scope), **skip the next two cells
("Setup" + "Inline helpers")** and jump to whichever Tier you want.

### B. Cold start
If you're in a fresh kernel, run the **Setup** + **Inline helpers** cells first.
They rebuild `_base`, the helpers, and the basis caches the Tier cells need.

Either way, each Tier cell asserts its dependencies at the top, so it'll fail loudly
rather than silently produce wrong numbers.


In [ ]:
# ============= SETUP (cold-start only — skip if kernel is warm from main notebook) =============
# Builds _base, _train_idx, _test_idx, raw connectome arrays, and the basis matrices.
# Identical to STEP 3/3.5/3.6/6.0 of the main notebook, condensed.
import importlib
import sys
from pathlib import Path

# Project root on sys.path so we can import main, models, data.
_REPO_ROOT = Path.cwd()
while _REPO_ROOT.name != "Conn2Conn" and _REPO_ROOT.parent != _REPO_ROOT:
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import main
import models.configs
import models.architectures
import models.eval.metrics as _metrics
from main import Sim

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

PARCELLATION = "Glasser"
DATA_LOAD_MODE = "precomputed"
CLOSED_FORM_MODELS = {
    "CrossModalPCA": {"config": _REPO_ROOT / "models/configs/CrossModalPCA.yml"},
}
SHUFFLE_SEED = 0

# Build Sim once.
_sim = Sim(
    model_name="CrossModalPCA",
    config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
    source="FC", target="SC",
    parcellation=PARCELLATION, shuffle_seed=SHUFFLE_SEED,
    data_load_mode=DATA_LOAD_MODE,
)
_base = _sim.base
_train_idx = _base.trainvaltest_partition_indices["train"]
_test_idx  = _base.trainvaltest_partition_indices["test"]

# Raw connectomes.
_SC_train = np.asarray(_base.sc_upper_triangles[_train_idx], dtype=np.float32)
_SC_test  = np.asarray(_base.sc_upper_triangles[_test_idx],  dtype=np.float32)
_FC_train = np.asarray(_base.fc_upper_triangles[_train_idx], dtype=np.float32)
_FC_test  = np.asarray(_base.fc_upper_triangles[_test_idx],  dtype=np.float32)

# Bases.
_X_brainvol_train = _base.fs_volumes_z[_train_idx]
_X_brainvol_test  = _base.fs_volumes_z[_test_idx]
_X_bv_train       = _X_brainvol_train
_X_bv_test        = _X_brainvol_test
_X_demo_train = np.concatenate([
    _base.age_z[_train_idx], _base.sex_oh[_train_idx], _base.race_eth_oh[_train_idx]
], axis=1).astype(np.float32)
_X_demo_test  = np.concatenate([
    _base.age_z[_test_idx],  _base.sex_oh[_test_idx],  _base.race_eth_oh[_test_idx]
], axis=1).astype(np.float32)
_X_bvdemo_train = np.concatenate([_X_bv_train, _X_demo_train], axis=1)
_X_bvdemo_test  = np.concatenate([_X_bv_test,  _X_demo_test ], axis=1)

_BASES = {
    "bv":      (_X_bv_train,     _X_bv_test,     _X_bv_train.shape[1]),
    "demo":    (_X_demo_train,   _X_demo_test,   _X_demo_train.shape[1]),
    "bv+demo": (_X_bvdemo_train, _X_bvdemo_test, _X_bvdemo_train.shape[1]),
}

print(f"Setup complete. train={len(_train_idx)}  test={len(_test_idx)}  parc={PARCELLATION}")
print(f"  bv:      {_X_bv_train.shape[1]}-dim")
print(f"  demo:    {_X_demo_train.shape[1]}-dim")
print(f"  bv+demo: {_X_bvdemo_train.shape[1]}-dim")


In [ ]:
# ============= INLINE HELPERS (cold-start only — skip if kernel is warm from main notebook) =============
# These are identical to STEP 6.0 + STEP 20 helpers in the main notebook.

def _fit_basis_ols(X_train, X_test, Y_train):
    """Multi-output OLS Y ~ X on train; returns (Y_pred_train, Y_pred_test)."""
    reg = LinearRegression().fit(X_train, Y_train)
    return (reg.predict(X_train).astype(np.float32),
            reg.predict(X_test ).astype(np.float32))


def _pca_pls_predict(X_src_train, X_src_test, Y_train,
                      k_src=256, k_tgt=256, k_pls=64, max_iter=2000):
    """PCA(src) + PCA(tgt) + PLS(latent) + inverse-PCA. Matches main notebook STEP 6.0."""
    pca_src = PCA(n_components=k_src, random_state=0).fit(X_src_train)
    pca_tgt = PCA(n_components=k_tgt, random_state=0).fit(Y_train)
    Z_src_tr = pca_src.transform(X_src_train)
    Z_src_te = pca_src.transform(X_src_test)
    Z_tgt_tr = pca_tgt.transform(Y_train)
    pls = PLSRegression(n_components=k_pls, scale=True, max_iter=max_iter).fit(Z_src_tr, Z_tgt_tr)
    Z_tgt_te_pred = pls.predict(Z_src_te)
    return pca_tgt.inverse_transform(Z_tgt_te_pred).astype(np.float32)


def _br_per_component_predict(X_src_train, X_src_test, Y_train,
                                k_src=256, k_tgt=256, max_iter=300):
    """PCA(src) + PCA(tgt) + BayesianRidge per target PCA component + inverse-PCA.
    Different inductive bias from PLS — per-component shrinkage vs shared latent."""
    pca_src = PCA(n_components=k_src, random_state=0).fit(X_src_train)
    pca_tgt = PCA(n_components=k_tgt, random_state=0).fit(Y_train)
    Z_src_tr = pca_src.transform(X_src_train)
    Z_src_te = pca_src.transform(X_src_test)
    Z_tgt_tr = pca_tgt.transform(Y_train)
    Z_tgt_te_pred = np.zeros((Z_src_te.shape[0], k_tgt), dtype=np.float32)
    for k in range(k_tgt):
        m = BayesianRidge(max_iter=max_iter).fit(Z_src_tr, Z_tgt_tr[:, k])
        Z_tgt_te_pred[:, k] = m.predict(Z_src_te)
    return pca_tgt.inverse_transform(Z_tgt_te_pred).astype(np.float32)


def _full_panel_eval(y_pred, y_true, target_train_mean_vec):
    """Returns dict with mse, r2, pearson, demeaned_pearson, top1_acc, avg_rank."""
    yp = np.asarray(y_pred, dtype=np.float32)
    yt = np.asarray(y_true, dtype=np.float32)
    mu = np.asarray(target_train_mean_vec, dtype=np.float32)
    cc_raw = _metrics.compute_corr_matrix(
        torch.tensor(yt, dtype=torch.float32),
        torch.tensor(yp, dtype=torch.float32),
    )
    if hasattr(cc_raw, "cpu"):
        cc_raw = cc_raw.cpu().numpy()
    panel = _metrics.compute_basic_regression_metrics(
        torch.tensor(yp, dtype=torch.float32),
        torch.tensor(yt, dtype=torch.float32),
        corr_matrix=torch.tensor(cc_raw, dtype=torch.float32),
        corr_matrix_demeaned=None,
    )
    if isinstance(panel, dict):
        panel = {k: float(v) if hasattr(v, "item") else float(v) for k, v in panel.items()}
    yp_dm = yp - mu
    yt_dm = yt - mu
    num   = (yp_dm * yt_dm).sum(axis=1)
    den_p = np.sqrt((yp_dm ** 2).sum(axis=1))
    den_t = np.sqrt((yt_dm ** 2).sum(axis=1))
    panel["demeaned_pearson"] = float((num / (den_p * den_t + 1e-10)).mean())
    return panel


# Residual cache (per basis × per target modality).
_RESID = {}
for _b, (_Xt, _Xe, _d) in _BASES.items():
    _RESID[_b] = {}
    for _tgt_name, _Y_tr, _Y_te in [("SC", _SC_train, _SC_test), ("FC", _FC_train, _FC_test)]:
        _pred_tr, _pred_te = _fit_basis_ols(_Xt, _Xe, _Y_tr)
        _RESID[_b][_tgt_name] = {
            "ols_pred_train": _pred_tr,
            "ols_pred_test":  _pred_te,
            "resid_train":    (_Y_tr - _pred_tr).astype(np.float32),
            "resid_test":     (_Y_te - _pred_te).astype(np.float32),
        }

print("Helpers + _RESID cache defined.")


---

## Tier 2.1 — 10-seed BR robustness for ALL bases

**The gap**: Exp 5 in the main notebook showed PLS=1.39× / BR=1.42× on `bv+demo` at
single seed — confirming model-class invariance for the stringent basis. STEP 11 then
extended PLS to 10 seeds across `none`, `bv`, `demo`, `bv+demo`. **Still missing**:
10-seed BR on `none`, `bv`, `demo` (only `bv+demo` was BR-tested, and only single seed).

**What this enables**: confirm the FC→SC > SC→FC asymmetry is model-class-invariant
across EVERY basis at paper-grade 10-seed precision. Currently the BR result is a
single-seed point estimate for one basis only.

**Cost**: ~50 min on a 4-CPU allocation. BR is the slow path — 256 fits per direction
× ~1 sec each = ~60 sec per BR call. 3 new bases × 2 directions × 2 frameworks × 10
seeds = 120 BR calls = ~2 hours total. Cache-resumable, so partial runs survive.

**Output**: `results/local_results/Tier2_1_BR_all_bases/seed_*.csv` + `aggregate.csv`.


In [ ]:
# ============= TIER 2.1 — 10-seed BR cross-modal on all bases =============
# Cache-resumable per-seed loop. Skip cells where the seed file already exists.
import numpy as _np
import pandas as _pd
import scipy.stats as _stats
from pathlib import Path as _Path

assert '_pca_pls_predict' in dir() and '_br_per_component_predict' in dir(), "Run helpers cell first."
assert '_full_panel_eval' in dir(), "Run helpers cell first."

_T21_OUT = _Path("results/local_results/Tier2_1_BR_all_bases")
_T21_OUT.mkdir(parents=True, exist_ok=True)

_T21_SEEDS = list(range(10))
_T21_BASES = ["none", "bv", "demo", "bv+demo"]
_T21_METRICS = ["mse", "r2", "pearson", "demeaned_pearson", "top1_acc", "avg_rank"]

print(f"Tier 2.1 — 10-seed BR cross-modal × {len(_T21_BASES)} bases × 2 frameworks × 2 dirs.")
print(f"Cache: {_T21_OUT}/seed_*.csv  (~2h total fresh; resumes if files exist)\n")

for _seed in _T21_SEEDS:
    _cache = _T21_OUT / f"seed_{_seed}.csv"
    if _cache.exists():
        print(f"  seed {_seed}: cached, skipping")
        continue
    print(f"--- seed {_seed} ---", flush=True)

    _sim = Sim(model_name="CrossModalPCA",
               config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
               source="FC", target="SC",
               parcellation=PARCELLATION, shuffle_seed=_seed,
               data_load_mode=DATA_LOAD_MODE)
    _b   = _sim.base
    _itr = _b.trainvaltest_partition_indices["train"]
    _ite = _b.trainvaltest_partition_indices["test"]

    _SC_tr = _np.asarray(_b.sc_upper_triangles[_itr], dtype=_np.float32)
    _SC_te = _np.asarray(_b.sc_upper_triangles[_ite], dtype=_np.float32)
    _FC_tr = _np.asarray(_b.fc_upper_triangles[_itr], dtype=_np.float32)
    _FC_te = _np.asarray(_b.fc_upper_triangles[_ite], dtype=_np.float32)
    _Xbv_tr = _b.fs_volumes_z[_itr]
    _Xbv_te = _b.fs_volumes_z[_ite]
    _Xdm_tr = _np.concatenate([_b.age_z[_itr], _b.sex_oh[_itr], _b.race_eth_oh[_itr]], axis=1).astype(_np.float32)
    _Xdm_te = _np.concatenate([_b.age_z[_ite], _b.sex_oh[_ite], _b.race_eth_oh[_ite]], axis=1).astype(_np.float32)
    _Xbd_tr = _np.concatenate([_Xbv_tr, _Xdm_tr], axis=1)
    _Xbd_te = _np.concatenate([_Xbv_te, _Xdm_te], axis=1)

    _resid_cache = {}
    for _basis, _X_tr, _X_te in [("bv", _Xbv_tr, _Xbv_te), ("demo", _Xdm_tr, _Xdm_te), ("bv+demo", _Xbd_tr, _Xbd_te)]:
        _sc_pred_tr, _sc_pred_te = _fit_basis_ols(_X_tr, _X_te, _SC_tr)
        _fc_pred_tr, _fc_pred_te = _fit_basis_ols(_X_tr, _X_te, _FC_tr)
        _resid_cache[(_basis, "SC")] = ((_SC_tr - _sc_pred_tr).astype(_np.float32), (_SC_te - _sc_pred_te).astype(_np.float32))
        _resid_cache[(_basis, "FC")] = ((_FC_tr - _fc_pred_tr).astype(_np.float32), (_FC_te - _fc_pred_te).astype(_np.float32))

    _rows = []
    for _basis in _T21_BASES:
        for _framework in ["target_only", "double_sided"]:
            if _basis == "none" and _framework == "double_sided":
                continue
            for _direction in ["FC->SC", "SC->FC"]:
                if _basis == "none":
                    _Xsrc_tr, _Xsrc_te = (_FC_tr, _FC_te) if _direction == "FC->SC" else (_SC_tr, _SC_te)
                else:
                    if _framework == "target_only":
                        _Xsrc_tr, _Xsrc_te = (_FC_tr, _FC_te) if _direction == "FC->SC" else (_SC_tr, _SC_te)
                    else:
                        _src_mod = "FC" if _direction == "FC->SC" else "SC"
                        _Xsrc_tr, _Xsrc_te = _resid_cache[(_basis, _src_mod)]
                _tgt_mod = "SC" if _direction == "FC->SC" else "FC"
                if _basis == "none":
                    _Y_tr, _Y_te = (_SC_tr, _SC_te) if _direction == "FC->SC" else (_FC_tr, _FC_te)
                else:
                    _Y_tr, _Y_te = _resid_cache[(_basis, _tgt_mod)]
                _mu = _Y_tr.mean(axis=0)

                _pred = _br_per_component_predict(_Xsrc_tr, _Xsrc_te, _Y_tr)
                _panel = _full_panel_eval(_pred, _Y_te, _mu)
                _row = {"seed": _seed, "basis": _basis, "framework": _framework,
                        "direction": _direction, "method": "BR_per_component"}
                _row.update({_m: _panel.get(_m, _np.nan) for _m in _T21_METRICS})
                _rows.append(_row)
                print(f"  ({_basis}, {_framework}, {_direction}) BR  demeaned_r = {_panel['demeaned_pearson']:.4f}", flush=True)

    _pd.DataFrame(_rows).to_csv(_cache, index=False)
    print(f"  saved {_cache}", flush=True)

# Aggregate across seeds + ratios.
_files = sorted(_T21_OUT.glob("seed_*.csv"))
print(f"\nLoading {len(_files)} per-seed files...")
_all = _pd.concat([_pd.read_csv(_f) for _f in _files], ignore_index=True)

_ratio_rows = []
for (_basis, _fw), _g in _all.groupby(["basis", "framework"]):
    _fs = _g[_g.direction == "FC->SC"].sort_values("seed")
    _sf = _g[_g.direction == "SC->FC"].sort_values("seed")
    if len(_fs) != len(_sf): continue
    _ratios = _fs["demeaned_pearson"].values / _np.maximum(_sf["demeaned_pearson"].values, 1e-12)
    _t1 = _stats.ttest_1samp(_ratios, 1.0) if len(_ratios) >= 2 else None
    _t115 = _stats.ttest_1samp(_ratios, 1.15) if len(_ratios) >= 2 else None
    _ratio_rows.append({
        "basis": _basis, "framework": _fw,
        "ratio_mean": float(_ratios.mean()), "ratio_std": float(_ratios.std(ddof=1)),
        "n_above_1.00": int((_ratios > 1.00).sum()),
        "n_above_1.15": int((_ratios > 1.15).sum()),
        "p_vs_1.00": float(_t1.pvalue) if _t1 else _np.nan,
        "p_vs_1.15": float(_t115.pvalue) if _t115 else _np.nan,
    })
_ratio_df = _pd.DataFrame(_ratio_rows)
_ratio_df.to_csv(_T21_OUT / "aggregate_ratios.csv", index=False)
print()
print("=== Tier 2.1 BR ratios (FC->SC / SC->FC, demeaned-r), 10-seed ===")
print(_ratio_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print(f"\nSaved aggregate_ratios.csv in {_T21_OUT}")


---

## Tier 3.4 — Cortical thickness as higher-rank anatomy basis

**The gap**: throughout the project, "anatomy" = 16 FreeSurfer brain-volume features.
Per [sanity check 1](sanity_checks.ipynb), that 16-feature OLS is 88% brain size and
only 12% compositional structure on the demeaned-r metric. Cortical thickness is
fundamentally different from volumes — regionally resolved per-parcel structural
variation. A 340-dim cortical thickness vector would test whether **anatomy as
conventionally measured (16 volumes) substantially undercounts anatomy's predictive
power for SC**.

**Four-outcome hypothesis space** (each implies a different writeup):

| `ct → SC` demeaned-r | Reading |
|---|---|
| ≈ 0.16 (same as bv) | composition saturates at 16 features; nothing to update |
| 0.20–0.25 | composition adds real signal; asymmetry ratio shrinks ~20-30% |
| 0.30–0.40 | regional cortex is a major SC predictor; major reframe of bv→SC = 0.167 baseline |
| ≥ 0.50 | anatomy almost completely explains SC; pivot rationale weakens |

**Prior**: I'd predict 0.20–0.30 (option B / low C) — cortical thickness has known
individual-difference structure, but SC reflects white-matter connectivity which is
related to but distinct from cortical thickness.

**Pre-requisite**: cortical thickness data must be accessible. The probe cell below
checks `_base.fs_features_all`. If cortical thickness isn't there, the experiment
needs upstream data loading work first.


In [ ]:
# ============= TIER 3.4 — STEP 1: probe for cortical thickness data on _base =============
# This cell does NO compute. It only checks whether cortical thickness is accessible
# in the current data pipeline.
assert '_base' in dir(), "Run Setup cell first."

print("Checking _base attributes for cortical thickness data...")
_attrs_to_check = ["fs_features_all", "fs_feature_columns", "cortical_thickness",
                   "ct_features", "fs_thickness", "fs_thickness_z"]
for _a in _attrs_to_check:
    if hasattr(_base, _a):
        _v = getattr(_base, _a)
        _shape = _v.shape if hasattr(_v, "shape") else len(_v) if hasattr(_v, "__len__") else "?"
        print(f"  ✓ {_a}: shape={_shape}, type={type(_v).__name__}")
    else:
        print(f"  ✗ {_a}: not present")

print()
if hasattr(_base, "fs_features_all"):
    print(f"fs_features_all has {_base.fs_features_all.shape[1]} columns.")
    if hasattr(_base, "fs_feature_columns"):
        print("First 30 column names:")
        for i, c in enumerate(_base.fs_feature_columns[:30]):
            print(f"  [{i:>3}] {c}")
        # Look for cortical-thickness-named columns.
        _ct_cols = [(i, c) for i, c in enumerate(_base.fs_feature_columns)
                    if any(k.lower() in c.lower() for k in ["thck", "thickness", "thick"])]
        print()
        print(f"Cortical-thickness-named columns found: {len(_ct_cols)}")
        for i, c in _ct_cols[:10]:
            print(f"  [{i:>3}] {c}")
        if len(_ct_cols) > 0:
            print()
            print("→ Cortical thickness IS available. Proceed to Tier 3.4 main cell.")
        else:
            print()
            print("→ Cortical thickness NOT in fs_features_all. Data loading work needed:")
            print("   1. Check HCP S1200 release for *.thickness.32k_fs_LR.dscalar.nii per subject")
            print("   2. Parcellate to Glasser → 340-dim per-parcel thickness vector")
            print("   3. Add as new attribute on HCP_Base (similar to fs_volumes_z pattern)")
    else:
        print("  fs_feature_columns attribute not present — can't identify which cols are thickness.")
else:
    print("fs_features_all not on _base. Data loading work needed (see notes in markdown above).")


In [ ]:
# ============= TIER 3.4 — STEP 2: cortical thickness experiment =============
# Runs only if the probe cell above found cortical thickness columns.
# Otherwise prints a "skipped, see notes" message and does nothing.
import numpy as _np
import pandas as _pd
from pathlib import Path as _Path
from sklearn.linear_model import LinearRegression as _LR

assert '_base' in dir(), "Run Setup cell first."
assert '_pca_pls_predict' in dir(), "Run helpers cell first."

_T34_OUT = _Path("results/local_results/Tier3_4_cortical_thickness")
_T34_OUT.mkdir(parents=True, exist_ok=True)

# Locate cortical thickness columns.
_ct_col_indices = []
if hasattr(_base, "fs_features_all") and hasattr(_base, "fs_feature_columns"):
    _ct_col_indices = [i for i, c in enumerate(_base.fs_feature_columns)
                       if any(k.lower() in c.lower() for k in ["thck", "thickness", "thick"])]

if len(_ct_col_indices) < 10:
    print(f"SKIP: only {len(_ct_col_indices)} cortical-thickness-named columns found.")
    print("Need >= ~50 for a meaningful higher-rank anatomy basis test. Data loading work needed first.")
else:
    print(f"Found {len(_ct_col_indices)} cortical thickness columns. Building _X_ct basis.")

    # Z-score on train.
    _ct_all = _base.fs_features_all[:, _ct_col_indices].astype(_np.float32)
    _ct_train_mean = _ct_all[_train_idx].mean(axis=0)
    _ct_train_std  = _ct_all[_train_idx].std(axis=0)
    _ct_train_std[_ct_train_std == 0] = 1.0
    _ct_z = (_ct_all - _ct_train_mean) / _ct_train_std
    _X_ct_train = _ct_z[_train_idx]
    _X_ct_test  = _ct_z[_test_idx]

    print(f"  _X_ct_train: {_X_ct_train.shape}, _X_ct_test: {_X_ct_test.shape}")

    # Baseline ct → SC, ct → FC.
    _rows = []
    for _label, _Y_tr, _Y_te, _mu_tr in [
        ("ct -> SC", _SC_train, _SC_test, _SC_train.mean(axis=0)),
        ("ct -> FC", _FC_train, _FC_test, _FC_train.mean(axis=0)),
    ]:
        _reg = _LR().fit(_X_ct_train, _Y_tr)
        _pred = _reg.predict(_X_ct_test).astype(_np.float32)
        _panel = _full_panel_eval(_pred, _Y_te, _mu_tr)
        _rows.append({"experiment": _label, "basis_dim": _X_ct_train.shape[1], **_panel})

    # Also bv+ct+demo as a higher-rank stringent basis.
    _X_bvctdemo_train = _np.concatenate([_X_bv_train, _X_ct_train, _X_demo_train], axis=1)
    _X_bvctdemo_test  = _np.concatenate([_X_bv_test,  _X_ct_test,  _X_demo_test ], axis=1)
    print(f"  bv+ct+demo basis: {_X_bvctdemo_train.shape[1]}-dim")

    for _label, _Y_tr, _Y_te, _mu_tr in [
        ("bv+ct+demo -> SC", _SC_train, _SC_test, _SC_train.mean(axis=0)),
        ("bv+ct+demo -> FC", _FC_train, _FC_test, _FC_train.mean(axis=0)),
    ]:
        _reg = _LR().fit(_X_bvctdemo_train, _Y_tr)
        _pred = _reg.predict(_X_bvctdemo_test).astype(_np.float32)
        _panel = _full_panel_eval(_pred, _Y_te, _mu_tr)
        _rows.append({"experiment": _label, "basis_dim": _X_bvctdemo_train.shape[1], **_panel})

    # Cross-modal asymmetry under bv+ct+demo residual basis.
    _SC_bvctdemo_pred_tr, _SC_bvctdemo_pred_te = _fit_basis_ols(_X_bvctdemo_train, _X_bvctdemo_test, _SC_train)
    _FC_bvctdemo_pred_tr, _FC_bvctdemo_pred_te = _fit_basis_ols(_X_bvctdemo_train, _X_bvctdemo_test, _FC_train)
    _SC_res_tr = (_SC_train - _SC_bvctdemo_pred_tr).astype(_np.float32)
    _SC_res_te = (_SC_test  - _SC_bvctdemo_pred_te).astype(_np.float32)
    _FC_res_tr = (_FC_train - _FC_bvctdemo_pred_tr).astype(_np.float32)
    _FC_res_te = (_FC_test  - _FC_bvctdemo_pred_te).astype(_np.float32)

    _pred_fs = _pca_pls_predict(_FC_train, _FC_test, _SC_res_tr)
    _pred_sf = _pca_pls_predict(_SC_train, _SC_test, _FC_res_tr)
    _panel_fs = _full_panel_eval(_pred_fs, _SC_res_te, _SC_res_tr.mean(axis=0))
    _panel_sf = _full_panel_eval(_pred_sf, _FC_res_te, _FC_res_tr.mean(axis=0))
    _ratio = _panel_fs["demeaned_pearson"] / max(_panel_sf["demeaned_pearson"], 1e-12)

    _rows.append({"experiment": "FC->SC_resid_bv+ct+demo (target-only PLS)", "basis_dim": _X_bvctdemo_train.shape[1], **_panel_fs})
    _rows.append({"experiment": "SC->FC_resid_bv+ct+demo (target-only PLS)", "basis_dim": _X_bvctdemo_train.shape[1], **_panel_sf})

    _df = _pd.DataFrame(_rows)
    _df.to_csv(_T34_OUT / "single_seed_results.csv", index=False)
    print()
    print("=== Tier 3.4 single-seed results ===")
    print(_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    print()
    print(f"Cross-modal asymmetry on bv+ct+demo residual (target-only PLS):")
    print(f"  FC->SC_resid  demeaned = {_panel_fs['demeaned_pearson']:.4f}")
    print(f"  SC->FC_resid  demeaned = {_panel_sf['demeaned_pearson']:.4f}")
    print(f"  ratio (FC->SC / SC->FC) = {_ratio:.3f}x")
    print()
    print(f"Compare to bv+demo (without ct, target-only): 1.39x")
    print(f"Saved single_seed_results.csv in {_T34_OUT}")
    print()
    print("Next step (if interesting): 10-seed extension following the STEP 11 pattern.")


---

## Tier 3.1 — K_PCA dimension sensitivity sweep

**The gap**: the project uses K_PCA = 256 everywhere (project default). We haven't
checked whether the headline 1.5× asymmetry ratio is stable across K_PCA values.

**Action**: re-run Phase 1's `bv+demo` target-only + double-sided 10-seed asymmetry at
K_PCA ∈ {128, 256, 512}. The middle value reproduces the existing result (sanity check);
the outliers test robustness.

**What this checks**: is the headline an artifact of K=256? Likely not — expectation
is the ratio is stable across K from ~64 onward — but reviewers ask.

**Cost**: ~10-15 min (2 alternates × 10 seeds × 4 PLS calls per seed × ~3 sec).
Outputs `results/local_results/Tier3_1_KPCA_sweep/sweep_results.csv`.


In [ ]:
# ============= TIER 3.1 — K_PCA sensitivity sweep =============
import numpy as _np
import pandas as _pd
from pathlib import Path as _Path

assert '_pca_pls_predict' in dir() and '_full_panel_eval' in dir(), "Run helpers cell first."

_T31_OUT = _Path("results/local_results/Tier3_1_KPCA_sweep")
_T31_OUT.mkdir(parents=True, exist_ok=True)

_K_PCA_VALUES = [128, 256, 512]
_SEEDS = list(range(10))
_rows = []

for _k_pca in _K_PCA_VALUES:
    print(f"\n=== K_PCA = {_k_pca} ===")
    for _seed in _SEEDS:
        _sim = Sim(model_name="CrossModalPCA",
                   config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
                   source="FC", target="SC",
                   parcellation=PARCELLATION, shuffle_seed=_seed,
                   data_load_mode=DATA_LOAD_MODE)
        _b = _sim.base
        _itr = _b.trainvaltest_partition_indices["train"]
        _ite = _b.trainvaltest_partition_indices["test"]
        _SC_tr = _np.asarray(_b.sc_upper_triangles[_itr], dtype=_np.float32)
        _SC_te = _np.asarray(_b.sc_upper_triangles[_ite], dtype=_np.float32)
        _FC_tr = _np.asarray(_b.fc_upper_triangles[_itr], dtype=_np.float32)
        _FC_te = _np.asarray(_b.fc_upper_triangles[_ite], dtype=_np.float32)
        _Xbd_tr = _np.concatenate([_b.fs_volumes_z[_itr], _b.age_z[_itr], _b.sex_oh[_itr], _b.race_eth_oh[_itr]], axis=1).astype(_np.float32)
        _Xbd_te = _np.concatenate([_b.fs_volumes_z[_ite], _b.age_z[_ite], _b.sex_oh[_ite], _b.race_eth_oh[_ite]], axis=1).astype(_np.float32)
        _SC_bd_pred_tr, _SC_bd_pred_te = _fit_basis_ols(_Xbd_tr, _Xbd_te, _SC_tr)
        _FC_bd_pred_tr, _FC_bd_pred_te = _fit_basis_ols(_Xbd_tr, _Xbd_te, _FC_tr)
        _SC_res_tr = (_SC_tr - _SC_bd_pred_tr).astype(_np.float32)
        _FC_res_tr = (_FC_tr - _FC_bd_pred_tr).astype(_np.float32)
        _SC_res_te = (_SC_te - _SC_bd_pred_te).astype(_np.float32)
        _FC_res_te = (_FC_te - _FC_bd_pred_te).astype(_np.float32)

        _pred_fs = _pca_pls_predict(_FC_tr, _FC_te, _SC_res_tr, k_src=_k_pca, k_tgt=_k_pca)
        _pred_sf = _pca_pls_predict(_SC_tr, _SC_te, _FC_res_tr, k_src=_k_pca, k_tgt=_k_pca)
        _p_fs = _full_panel_eval(_pred_fs, _SC_res_te, _SC_res_tr.mean(axis=0))
        _p_sf = _full_panel_eval(_pred_sf, _FC_res_te, _FC_res_tr.mean(axis=0))
        _ratio = _p_fs["demeaned_pearson"] / max(_p_sf["demeaned_pearson"], 1e-12)
        _rows.append({"k_pca": _k_pca, "seed": _seed,
                      "FC->SC_demeaned": _p_fs["demeaned_pearson"],
                      "SC->FC_demeaned": _p_sf["demeaned_pearson"],
                      "ratio": _ratio})
        print(f"  seed {_seed}: FC->SC = {_p_fs['demeaned_pearson']:.4f}, SC->FC = {_p_sf['demeaned_pearson']:.4f}, ratio = {_ratio:.3f}x")

_df = _pd.DataFrame(_rows)
_df.to_csv(_T31_OUT / "sweep_results.csv", index=False)

# Aggregate by K_PCA
print()
print("=== Tier 3.1 K_PCA sensitivity (bv+demo target-only, 10-seed) ===")
_agg = _df.groupby("k_pca").agg(
    ratio_mean=("ratio", "mean"),
    ratio_std=("ratio", "std"),
    n_above_115=("ratio", lambda x: (x > 1.15).sum()),
).reset_index()
print(_agg.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print(f"\nSaved sweep_results.csv in {_T31_OUT}")


---

## Tier 3.2 — K_PLS components sensitivity sweep

**The gap**: K_PLS = 64 everywhere. Model-flexibility sensitivity not checked.

**Action**: re-run the same `bv+demo` target-only 10-seed asymmetry at K_PLS ∈ {32, 64, 128}.

**Cost**: ~10 min. Output `results/local_results/Tier3_2_KPLS_sweep/sweep_results.csv`.


In [ ]:
# ============= TIER 3.2 — K_PLS components sensitivity sweep =============
import numpy as _np
import pandas as _pd
from pathlib import Path as _Path

assert '_pca_pls_predict' in dir() and '_full_panel_eval' in dir(), "Run helpers cell first."

_T32_OUT = _Path("results/local_results/Tier3_2_KPLS_sweep")
_T32_OUT.mkdir(parents=True, exist_ok=True)

_K_PLS_VALUES = [32, 64, 128]
_SEEDS = list(range(10))
_rows = []

for _k_pls in _K_PLS_VALUES:
    print(f"\n=== K_PLS = {_k_pls} ===")
    for _seed in _SEEDS:
        _sim = Sim(model_name="CrossModalPCA",
                   config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
                   source="FC", target="SC",
                   parcellation=PARCELLATION, shuffle_seed=_seed,
                   data_load_mode=DATA_LOAD_MODE)
        _b = _sim.base
        _itr = _b.trainvaltest_partition_indices["train"]
        _ite = _b.trainvaltest_partition_indices["test"]
        _SC_tr = _np.asarray(_b.sc_upper_triangles[_itr], dtype=_np.float32)
        _SC_te = _np.asarray(_b.sc_upper_triangles[_ite], dtype=_np.float32)
        _FC_tr = _np.asarray(_b.fc_upper_triangles[_itr], dtype=_np.float32)
        _FC_te = _np.asarray(_b.fc_upper_triangles[_ite], dtype=_np.float32)
        _Xbd_tr = _np.concatenate([_b.fs_volumes_z[_itr], _b.age_z[_itr], _b.sex_oh[_itr], _b.race_eth_oh[_itr]], axis=1).astype(_np.float32)
        _Xbd_te = _np.concatenate([_b.fs_volumes_z[_ite], _b.age_z[_ite], _b.sex_oh[_ite], _b.race_eth_oh[_ite]], axis=1).astype(_np.float32)
        _SC_bd_pred_tr, _SC_bd_pred_te = _fit_basis_ols(_Xbd_tr, _Xbd_te, _SC_tr)
        _FC_bd_pred_tr, _FC_bd_pred_te = _fit_basis_ols(_Xbd_tr, _Xbd_te, _FC_tr)
        _SC_res_tr = (_SC_tr - _SC_bd_pred_tr).astype(_np.float32)
        _FC_res_tr = (_FC_tr - _FC_bd_pred_tr).astype(_np.float32)
        _SC_res_te = (_SC_te - _SC_bd_pred_te).astype(_np.float32)
        _FC_res_te = (_FC_te - _FC_bd_pred_te).astype(_np.float32)

        _pred_fs = _pca_pls_predict(_FC_tr, _FC_te, _SC_res_tr, k_pls=_k_pls)
        _pred_sf = _pca_pls_predict(_SC_tr, _SC_te, _FC_res_tr, k_pls=_k_pls)
        _p_fs = _full_panel_eval(_pred_fs, _SC_res_te, _SC_res_tr.mean(axis=0))
        _p_sf = _full_panel_eval(_pred_sf, _FC_res_te, _FC_res_tr.mean(axis=0))
        _ratio = _p_fs["demeaned_pearson"] / max(_p_sf["demeaned_pearson"], 1e-12)
        _rows.append({"k_pls": _k_pls, "seed": _seed,
                      "FC->SC_demeaned": _p_fs["demeaned_pearson"],
                      "SC->FC_demeaned": _p_sf["demeaned_pearson"],
                      "ratio": _ratio})
        print(f"  seed {_seed}: ratio = {_ratio:.3f}x")

_df = _pd.DataFrame(_rows)
_df.to_csv(_T32_OUT / "sweep_results.csv", index=False)

print()
print("=== Tier 3.2 K_PLS sensitivity (bv+demo target-only, 10-seed) ===")
_agg = _df.groupby("k_pls").agg(
    ratio_mean=("ratio", "mean"),
    ratio_std=("ratio", "std"),
    n_above_115=("ratio", lambda x: (x > 1.15).sum()),
).reset_index()
print(_agg.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print(f"\nSaved sweep_results.csv in {_T32_OUT}")


---

## Tier 3.3 — Demographics decomposition

**The gap**: the `demo` basis combines age + sex + race_eth into a single 10-dim vector.
We don't know which component is doing the most predictive work. `demo → FC = 0.114`
is impressive — but is it mostly age (which correlates with global FC variance) or
sex/race_eth?

**Action**: also run with `age-only` (1 dim), `sex-only` (2 dim), `race_eth-only`
(~7 dim) as separate residualization bases. Check the demeaned-r for each on SC and FC,
and the resulting cross-modal asymmetry.

**Cost**: ~15 min. Output `results/local_results/Tier3_3_demo_decomp/results.csv`.


In [ ]:
# ============= TIER 3.3 — Demographics decomposition =============
import numpy as _np
import pandas as _pd
from pathlib import Path as _Path

assert '_pca_pls_predict' in dir() and '_full_panel_eval' in dir(), "Run helpers cell first."

_T33_OUT = _Path("results/local_results/Tier3_3_demo_decomp")
_T33_OUT.mkdir(parents=True, exist_ok=True)

_SEEDS = list(range(10))
_rows = []

for _seed in _SEEDS:
    _sim = Sim(model_name="CrossModalPCA",
               config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
               source="FC", target="SC",
               parcellation=PARCELLATION, shuffle_seed=_seed,
               data_load_mode=DATA_LOAD_MODE)
    _b = _sim.base
    _itr = _b.trainvaltest_partition_indices["train"]
    _ite = _b.trainvaltest_partition_indices["test"]
    _SC_tr = _np.asarray(_b.sc_upper_triangles[_itr], dtype=_np.float32)
    _SC_te = _np.asarray(_b.sc_upper_triangles[_ite], dtype=_np.float32)
    _FC_tr = _np.asarray(_b.fc_upper_triangles[_itr], dtype=_np.float32)
    _FC_te = _np.asarray(_b.fc_upper_triangles[_ite], dtype=_np.float32)

    # Three sub-bases.
    _bases = {
        "age":      (_b.age_z[_itr],      _b.age_z[_ite]),
        "sex":      (_b.sex_oh[_itr],     _b.sex_oh[_ite]),
        "race_eth": (_b.race_eth_oh[_itr], _b.race_eth_oh[_ite]),
        "demo_full":(_np.concatenate([_b.age_z[_itr], _b.sex_oh[_itr], _b.race_eth_oh[_itr]], axis=1).astype(_np.float32),
                     _np.concatenate([_b.age_z[_ite], _b.sex_oh[_ite], _b.race_eth_oh[_ite]], axis=1).astype(_np.float32)),
    }
    for _basis_name, (_Xb_tr, _Xb_te) in _bases.items():
        # Baseline → SC and → FC
        for _tgt_name, _Y_tr, _Y_te in [("SC", _SC_tr, _SC_te), ("FC", _FC_tr, _FC_te)]:
            _pred_tr, _pred_te = _fit_basis_ols(_Xb_tr, _Xb_te, _Y_tr)
            _panel = _full_panel_eval(_pred_te, _Y_te, _Y_tr.mean(axis=0))
            _rows.append({"seed": _seed, "basis": _basis_name, "target": _tgt_name,
                          "type": "baseline",
                          "demeaned_pearson": _panel["demeaned_pearson"],
                          "top1_acc": _panel["top1_acc"],
                          "avg_rank": _panel["avg_rank"]})
        # Cross-modal asymmetry on residual (target-only)
        _SC_res_tr = (_SC_tr - _fit_basis_ols(_Xb_tr, _Xb_te, _SC_tr)[0]).astype(_np.float32)
        _SC_res_te = (_SC_te - _fit_basis_ols(_Xb_tr, _Xb_te, _SC_tr)[1]).astype(_np.float32)
        _FC_res_tr = (_FC_tr - _fit_basis_ols(_Xb_tr, _Xb_te, _FC_tr)[0]).astype(_np.float32)
        _FC_res_te = (_FC_te - _fit_basis_ols(_Xb_tr, _Xb_te, _FC_tr)[1]).astype(_np.float32)
        _pred_fs = _pca_pls_predict(_FC_tr, _FC_te, _SC_res_tr)
        _pred_sf = _pca_pls_predict(_SC_tr, _SC_te, _FC_res_tr)
        _p_fs = _full_panel_eval(_pred_fs, _SC_res_te, _SC_res_tr.mean(axis=0))
        _p_sf = _full_panel_eval(_pred_sf, _FC_res_te, _FC_res_tr.mean(axis=0))
        _ratio = _p_fs["demeaned_pearson"] / max(_p_sf["demeaned_pearson"], 1e-12)
        _rows.append({"seed": _seed, "basis": _basis_name, "target": "ratio",
                      "type": "cross_modal_ratio",
                      "demeaned_pearson": _ratio,
                      "FC->SC_resid": _p_fs["demeaned_pearson"],
                      "SC->FC_resid": _p_sf["demeaned_pearson"]})
    print(f"  seed {_seed}: done")

_df = _pd.DataFrame(_rows)
_df.to_csv(_T33_OUT / "results.csv", index=False)

# Aggregate baselines by (basis, target)
print()
print("=== Tier 3.3 — Demographics decomposition, 10-seed mean ===")
print("Baselines (demeaned-r):")
_bl = _df[_df.type == "baseline"].groupby(["basis", "target"])["demeaned_pearson"].agg(["mean", "std"]).reset_index()
print(_bl.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print()
print("Cross-modal ratio (FC->SC / SC->FC) per residualization basis:")
_cm = _df[_df.type == "cross_modal_ratio"].groupby("basis")["demeaned_pearson"].agg(["mean", "std", lambda x: (x > 1.15).sum()]).reset_index()
_cm.columns = ["basis", "ratio_mean", "ratio_std", "n_above_1.15"]
print(_cm.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print(f"\nSaved results.csv in {_T33_OUT}")


---

## Tier 4 — Generalization (data-dependent)

_Markdown stub only — requires upstream data setup not in the current repo._

**The gap**: all results use the HCP-Young Adult cohort + Glasser parcellation. We
haven't shown the headline asymmetry is robust to:

- **4S456Parcels parcellation** (different cortical atlas, larger N per region). The
  raw data exists in the project tree; just needs parcellation toggle in `Sim(...)`.
  Estimate: ~2-3 hours to rerun the key cells. Worth doing once the writeup is locked.
- **HCP-Aging / HCP-Development cohorts** (older / younger samples). Tests lifespan
  generalization. Requires new HCP_Base loader paths; significant data-prep work
  outside this notebook's scope.

**Defer until**: pivot direction is locked AND a reviewer asks "but does this hold on
[X]?" — at which point either re-run on 4S456Parcels (cheap) or set up the new cohort
loader (expensive).


---

## Tier 5 — Methodology refinements

_Markdown stub only — not paper-blocking, polish for camera-ready / revisions._

**Three known methodological caveats** that could be tightened but don't change the
headline numbers materially:

1. **Cross-fit train predictions in Phase 2 Analysis 2.** Currently
   `_phase2_build_inputs_for_seed` uses `_pca_pls_predict(_FC_tr, _FC_tr, _SC_tr)` —
   the same PLS fit predicting in-sample on train. Slightly optimistic but applied
   consistently across all rows. To remove the optimism: implement K-fold cross-fit
   for train predictions. ~30 min implementation + ~30 min re-run.

2. **Family-structure AUC: per-seed AUCs with hierarchical CIs.** Current approach pools
   all pairs across 10 seeds then bootstraps. Alternative: per-seed AUC then mean ± CI
   across seeds. Slightly different statistical semantics; some statisticians prefer
   the per-seed approach for cross-validated AUCs. ~10 min change.

3. **Alternate anatomy basis: cortical thickness or surface area.** Tier 3.4 covers
   cortical thickness specifically. Surface area is a separate channel that could be
   added if Tier 3.4 reveals interesting cortex-structure dependencies. ~30 min once
   the data is set up.

**Defer until**: a specific reviewer comment makes one of these the unblocker. None
are load-bearing for the current paper-grade claims.
